# CCD Characterisation

`author` Julien Morin <julien.morin@umontpellier.fr>

`date` 11 Oct 2023

Reference: *Handbook of CCD Astronomy*, §4.3
* compute CCD gain and readout noise
* compare with specifications

In [6]:
# import and settings
import warnings

import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from astropy.stats import mad_std
from astropy import units as u
from astropy.visualization import ZScaleInterval
from astropy.wcs import FITSFixedWarning
import ccdproc as ccdp

warnings.simplefilter('ignore', FITSFixedWarning)

In [2]:
# create raw image collection
ifc_raw = ccdp.ImageFileCollection('photometry_raw_data_extracted/raw/')

## CCD gain and redout noise

In [14]:
# Load two bias and two flat-field frames
fnamel_bias = ifc_raw.files_filtered(imagetyp='Bias Frame', include_path=True)
print(fnamel_bias[0:2])
ccd_bias1 = ccdp.CCDData.read(fnamel_bias[0], unit='adu')
ccd_bias2 = ccdp.CCDData.read(fnamel_bias[1], unit='adu')
fnamel_flat = ifc_raw.files_filtered(imagetyp='Flat Field', include_path=True)
print(fnamel_flat[0:2])
ccd_flat1 = ccdp.CCDData.read(fnamel_flat[0], unit='adu')
ccd_flat2 = ccdp.CCDData.read(fnamel_flat[1], unit='adu')

['photometry_raw_data_extracted/raw/bias-0001.fits', 'photometry_raw_data_extracted/raw/bias-0002.fits']
['photometry_raw_data_extracted/raw/flat-0001_H.fits', 'photometry_raw_data_extracted/raw/flat-0002_H.fits']


In [15]:
# Be cautious when performing arithmetic operations on data, check you results!
## print basic stats of flats 1&2
print("flat 1: mean=%.2f  std=%.2f" % (np.mean(ccd_flat1.data), np.std(ccd_flat1.data)))
print("flat 2: mean=%.2f  std=%.2f" % (np.mean(ccd_flat2.data), np.std(ccd_flat2.data)))

print('\nexample 1 -- WRONG: naively subtract flats 1&2 data array')
diff_f1f2 = ccd_flat1.data - ccd_flat2.data
print("f1-f2: mean=%.2f  std=%.2f" % (np.mean(diff_f1f2), np.std(diff_f1f2)))
print(diff_f1f2.dtype)

print('\nexample 2: subtract flats 1&2 data arrays w/ data type conversion')
diff_f1f2 = ccd_flat1.data.astype('float32') - ccd_flat2.data.astype('float32')
print("f1-f2: mean=%.2f  std=%.2f" % (np.mean(diff_f1f2), np.std(diff_f1f2)))
print(diff_f1f2.dtype)

print('\nexample 3: subtract flats 1&2 CCDData objects w/ the specific method')
diff_f1f2 = ccd_flat1.subtract(ccd_flat2)
print("f1-f2: mean=%.2f  std=%.2f" % (np.mean(diff_f1f2.data), np.std(diff_f1f2.data)))
print(diff_f1f2.dtype)

# print the CCDData structure is kept in this case, but the metadata is not propagated
print(diff_f1f2.meta)


flat 1: mean=24563.24  std=1106.02
flat 2: mean=24565.40  std=1101.71

example 1 -- WRONG: naively subtract flats 1&2 data array
f1-f2: mean=32920.02  std=32589.46
uint16

example 2: subtract flats 1&2 data arrays w/ data type conversion
f1-f2: mean=-2.15  std=224.37
float32

example 3: subtract flats 1&2 CCDData objects w/ the specific method
f1-f2: mean=32920.02  std=32589.46
uint16
OrderedDict()


for more details about CCDData arithmetics see:
* https://ccdproc.readthedocs.io/en/latest/ccddata.html#arithmetic-with-images
* https://docs.astropy.org/en/stable/api/astropy.nddata.CCDData.html#astropy.nddata.CCDData §Methods Summary

In [25]:
### ToDo: print statistics of each individual image

# import and settings
import warnings

import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from astropy.stats import mad_std
from astropy import units as u
from astropy.visualization import ZScaleInterval
from astropy.wcs import FITSFixedWarning
import ccdproc as ccdp

warnings.simplefilter('ignore', FITSFixedWarning)

# CUSTOM FUNCTIONS

def imstat(ccd, fname):
    print(f"File : {fname}")
    print(f"Mean : {np.mean(ccd.data)}")
    print(f"Median : {np.median(ccd.data)}")
    print(f"Std : {np.std(ccd.data)}")

def read_float(path, unit='adu'):
    ccd = ccdp.CCDData.read(path, unit=unit)
    ccd.data = ccd.data.astype('float32')
    return ccd

# create raw image collection
ifc_raw = ccdp.ImageFileCollection('photometry_raw_data_extracted/raw/')

print(set(ifc_raw.summary['imagetyp'])) # Check what imagetyp are in the collection


fnamel_bias = ifc_raw.files_filtered(imagetyp='Bias Frame', include_path=True)
fnamel_flat = ifc_raw.files_filtered(imagetyp='Flat Field', include_path=True)

# WRONG, fnamel only carry the names (and path, since include_path=True), but no actual data)
bias_1, bias_2 = fnamel_bias[0], fnamel_bias[1] 
flat_1, flat_2 = fnamel_flat[0], fnamel_flat[1]

ccd_bias1 = read_float(fnamel_bias[0])
ccd_bias2 = read_float(fnamel_bias[1])

ccd_flat1 = read_float(fnamel_flat[0])
ccd_flat2 = read_float(fnamel_flat[1])

imstat(ccd_bias1, "ccd_bias1")
print(ccd_bias1.header)
imstat(ccd_bias2, "ccd_bias2")
print(ccd_bias2.header)

imstat(ccd_flat1, "ccd_flat1")
print(ccd_bias1.header)
imstat(ccd_flat2, "ccd_flat2")
print(ccd_flat2.header)



{np.str_('Bias Frame'), np.str_('Flat Field'), np.str_('Light Frame')}
File : ccd_bias1
Mean : 301.2579040527344
Median : 301.0
Std : 6.709732532501221
SIMPLE  =                    T                                                  BITPIX  =                   16 /8 unsigned int, 16 & 32 int, -32 & -64 real     NAXIS   =                    2 /number of axes                                  NAXIS1  =                 2048 /fastest changing axis                           NAXIS2  =                 2048 /next to fastest changing axis                   BSCALE  =   1.0000000000000000 /physical = BZERO + BSCALE*array_value           BZERO   =   32768.000000000000 /physical = BZERO + BSCALE*array_value           DATE-OBS= '2020-09-15T20:20:17.39' /YYYY-MM-DDThh:mm:ss observation start, UT   EXPTIME =  0.00000000000000000 /Exposure time in seconds                        EXPOSURE=  0.00000000000000000 /Exposure time in seconds                        SET-TEMP=  -60.000000000000000 /CCD temperature 

In [27]:
### ToDo: compute sums and differences of bias and flat-field frames and print statistics
diff_b1b2 = ccd_bias1.subtract(ccd_bias2)
imstat(diff_b1b2, "diff_b1b2")
print(diff_b1b2.header)

diff_f1f2 = ccd_flat1.subtract(ccd_flat2)
imstat(diff_f1f2, "diff_f1f2")
print(diff_f1f2.header)

File : diff_b1b2
Mean : 2.03482723236084
Median : 2.0
Std : 9.496003150939941
OrderedDict()
File : diff_f1f2
Mean : -2.15269136428833
Median : -2.0
Std : 224.36643981933594
OrderedDict()


In [32]:
### ToDo: Compute gain and readout noise using the formulas given in lecture #1

gain = (np.mean(ccd_flat1.data) + np.mean(ccd_flat2.data) - (np.mean(ccd_bias1.data) + np.mean(ccd_bias2.data)))/(np.var(diff_f1f2.data) - np.var(diff_b1b2.data))
print(gain)

rn = gain*np.std(diff_b1b2.data)/np.sqrt(2)
print(rn)

0.9657323
6.484591380263292


Compare your results with the specifications given on the OHP website : __[http://www.obs-hp.fr/guide/camera-120/camera-120.shtml#Detec](http://www.obs-hp.fr/guide/camera-120/camera-120.shtml#Detec)__